# 参数配置


In [ ]:
# data path
BASE_DATA_DIR = "./68 model"  # 数据根目录（包含68 model和导出的标注）
EXPORTED_ANNO_DIR = "exported_coco17_strict1"  # COCO格式JSON所在子目录
VIEW_NAMES = [f"{i:02d} visual_view" for i in range(1, 6)]  # ["01 visual_view", ..., "05 visual_view"]
# val
print(VIEW_NAMES)

# 图像文件名模板：frame_id 会被零填充，例如 frame_0001.png 对应 "{:04d}.png" 或 "frame_{:04d}.png"
IMG_NAME_TEMPLATES = ["{:04d}" + f"_{view_name[1]}.png" for view_name in VIEW_NAMES]
print(f"图像文件名模板: {IMG_NAME_TEMPLATES}")


# output 
COCO_JSON_PREFIX = "./anime_coco"  # 生成的COCO标注前缀（会自动加_train/_val.json）
WORK_DIR = "./work_dirs/anime_rtmpose_tiny"  # 训练工作目录（保存日志和权重）
TRAIN_VAL_SPLIT = 0.85  # 训练集比例（0.85 = 85%训练，15%验证，建议保留一些做验证）

# models (RTMPose family)
# 可选: "rtmpose-t_8xb256-420e_coco-256x192.py" (tiny,  fastest)
#       "rtmpose-s_8xb256-420e_coco-256x192.py" (small)
#       "rtmpose-m_8xb256-420e_coco-256x192.py" (medium)
# 注意：这是 mmpose/configs/body_2d_keypoint/rtmpose/coco/ 下的文件名
MODEL_CONFIG_NAME = "rtmpose-m_8xb256-420e_coco-256x192.py"

# 训练超参数
MAX_EPOCHS = 30          # 验证可行性建议20-30，实际生产建议100+
BATCH_SIZE = 8           # Tiny模型在8GB显存下可设为8-16
LR = 5e-4                # 学习率，小数据集建议3e-4~5e-4
VAL_INTERVAL = 5         # 每N个epoch验证并保存一次
# INPUT_SIZE = (256, 192)  # RTMPose-tiny默认输入尺寸 (H, W)
INPUT_SIZE = (192, 256)

DEVICE = 'cuda:0'        # or 'cpu'
RANDOM_SEED = 42

# COCO 17关键点标准顺序，非必要勿修改
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

['01 visual_view', '02 visual_view', '03 visual_view', '04 visual_view', '05 visual_view']
图像文件名模板: ['{:04d}_1.png', '{:04d}_2.png', '{:04d}_3.png', '{:04d}_4.png', '{:04d}_5.png']


In [1]:
import os
import sys
import json
import glob
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

# MMPose setup
if os.path.exists('./mmpose'):
    sys.path.insert(0, os.path.abspath('./mmpose'))
else: print("mmpose directory not found. please check the path or clone the repo to current directory.")
# MMEngine & MMPose 导入
from mmengine import Config
from mmengine.runner import Runner
from mmengine.dist import get_dist_info
from mmpose.utils import register_all_modules
from mmpose.datasets.datasets.base import BaseCocoStyleDataset
from mmpose.datasets import CocoDataset
from PIL import Image
print("环境初始化完成，MMPose版本检查通过。")
register_all_modules()

/home/tvem/anaconda3/envs/mmpose/lib/python3.8/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/home/tvem/anaconda3/envs/mmpose/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


环境初始化完成，MMPose版本检查通过。


# 转换为标准COCO格式
把可视化notebook输出的符合coco标准的17个点转换为coco标准数据结构用于fineturning
- 已划分训练集验证集。 
- train: `./anime_coco_train.json`
- validation: `./anime_coco_val.json`
- 

In [ ]:
def get_image_size(base_dir, img_rel_path):
    """获取图像真实尺寸"""
    try:
        with Image.open(os.path.join(base_dir, img_rel_path)) as img:
            return img.size  # (width, height)
    except Exception as e:
        return None

def convert_and_split_data():
    """
    【主函数】数据转换入口，内部使用 Cell 1 的全局参数
    返回: (train_anno_file, val_anno_file)
    """
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    
    # COCO模板
    coco_template = {
        "info": {"description": "AnimePose2D COCO Format", "version": "1.0"},
        "images": [],
        "annotations": [],
        "categories": [{
            "supercategory": "person", "id": 1, "name": "person",
            "keypoints": KEYPOINT_NAMES,
            "skeleton": [[16,14],[14,12],[17,15],[15,13],[12,13],[6,12],[7,13],[6,7],
                        [6,8],[7,9],[8,10],[9,11],[2,3],[1,2],[1,3],[2,4],[3,5],[4,6],[5,7]]
        }]
    }
    
    all_samples = []
    img_id = 1
    ann_id = 1
    
    print("扫描标注文件...")
    for idx, view_name in enumerate(VIEW_NAMES):
        view_prefix = view_name.split()[0]
        anno_dir = os.path.join(BASE_DATA_DIR, EXPORTED_ANNO_DIR, view_name)
        img_dir_rel = os.path.join(view_name, f"{view_prefix}_1 Rendering")
        
        if not os.path.exists(anno_dir):
            continue
            
        json_files = sorted(glob.glob(os.path.join(anno_dir, "*.json")))
        
        for json_path in tqdm(json_files, desc=f"{view_name}"):
            # 读取JSON
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
            except:
                continue
            
            # 定位图像
            frame_id = data.get("frame_id", 0)
            img_filename = IMG_NAME_TEMPLATES[idx].format(frame_id)
            img_rel_path = os.path.join(img_dir_rel, img_filename).replace("\\", "/")
            
            # 获取真实图像尺寸（关键！）
            img_size = get_image_size(BASE_DATA_DIR, img_rel_path)
            if img_size is None:
                # 尝试其他扩展名
                base = img_rel_path.rsplit('.', 1)[0]
                for ext in ['.jpg', '.jpeg', '.png']:
                    img_size = get_image_size(BASE_DATA_DIR, base + ext)
                    if img_size:
                        img_rel_path = base + ext
                        break
                if not img_size:
                    continue
            
            img_w, img_h = img_size
            
            # 解析关键点
            kpts = data.get("keypoints", {})
            keypoints, xs, ys = [], [], []
            for kp_name in KEYPOINT_NAMES:
                if kp_name in kpts:
                    x, y = float(kpts[kp_name]["x"]), float(kpts[kp_name]["y"])
                    keypoints.extend([x, y, 2])
                    xs.append(x)
                    ys.append(y)
                else:
                    keypoints.extend([0, 0, 0])
            
            # 计算 BBox
            if xs:
                min_x, max_x, min_y, max_y = min(xs), max(xs), min(ys), max(ys)
                w, h = max_x - min_x, max_y - min_y
                bbox = [max(0, min_x-w*0.05), max(0, min_y-h*0.05), w*1.1, h*1.1]
                area, num_kpts = bbox[2]*bbox[3], sum(1 for i in range(2,51,3) if keypoints[i]>0)
            else:
                bbox, area, num_kpts = [0,0,0,0], 0, 0
            
            all_samples.append({
                "id": img_id,  # COCO标准：图像主键为 "id"
                "file_name": img_rel_path,
                "width": int(img_w),
                "height": int(img_h),
                "ann_id": ann_id,
                "bbox": bbox,
                "area": area,
                "keypoints": keypoints,
                "num_keypoints": num_kpts
            })
            img_id += 1
            ann_id += 1
    
    # 划分训练/验证
    random.shuffle(all_samples)
    split_idx = int(len(all_samples) * TRAIN_VAL_SPLIT)
    train_s, val_s = all_samples[:split_idx], all_samples[split_idx:]
    
    # 保存函数
    def save_coco(samples, suffix):
        coco = json.loads(json.dumps(coco_template))
        for s in samples:
            coco["images"].append({
                "id": s["id"],
                "file_name": s["file_name"],
                "height": s["height"],
                "width": s["width"]
            })
            coco["annotations"].append({
                "id": s["ann_id"],
                "image_id": s["id"],  # 外键
                "category_id": 1,
                "bbox": s["bbox"],
                "area": s["area"],
                "keypoints": s["keypoints"],
                "num_keypoints": s["num_keypoints"],
                "iscrowd": 0,
                "segmentation": []  # mmpose必需
            })
        path = os.path.abspath(f"{COCO_JSON_PREFIX}_{suffix}.json")
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(coco, f, indent=2)
        return path
    
    train_path = save_coco(train_s, "train")
    val_path = save_coco(val_s, "val")
    
    print(f"\n✓ 生成完成: 训练{len(train_s)}张, 验证{len(val_s)}张")
    return train_path, val_path

# 执行转换并赋值给全局变量
train_anno_file, val_anno_file = convert_and_split_data()
print(f"训练集: {train_anno_file}")
print(f"验证集: {val_anno_file}")

✓ 环境初始化完成
扫描标注文件...


01 visual_view:   0%|          | 0/72 [00:00<?, ?it/s]

05 visual_view: 100%|██████████| 72/72 [00:00<00:00, 3707.08it/s]


✓ 生成完成: 训练306张, 验证54张
训练集: /home/tvem/anime/AniPose/anime_coco_train.json
验证集: /home/tvem/anime/AniPose/anime_coco_val.json


# 训练元数据定义
config

In [4]:
from mmpose.datasets import CocoDataset

# 加载基础配置
config_full_path = os.path.join(
    './mmpose/configs/body_2d_keypoint/rtmpose/coco', 
    MODEL_CONFIG_NAME
)

if not os.path.exists(config_full_path):
    raise FileNotFoundError(f"未找到配置文件: {config_full_path}")

cfg = Config.fromfile(config_full_path)

# ================= 关键参数覆盖区域 =================
cfg.work_dir = WORK_DIR
cfg.device = DEVICE

# 1. 设置 mmpose 配置根目录（用于解析 metainfo 相对路径）
mmpose_config_root = os.path.abspath('./mmpose/configs')
metainfo_file = os.path.join(mmpose_config_root, '_base_/datasets/coco.py')

# 验证文件存在（诊断用）
if not os.path.exists(metainfo_file):
    raise FileNotFoundError(
        f"找不到 metainfo 文件: {metainfo_file}\n"
        f"请确认: 1) mmpose 已完整克隆  2) 路径 ./mmpose/configs/_base_/datasets/coco.py 存在"
    )
print(f"✓ Metainfo 文件已定位: {metainfo_file}")

# 2. 数据集配置（使用绝对路径的 metainfo）
cfg.data_root = os.path.abspath(BASE_DATA_DIR)

# 训练集
cfg.train_dataloader.dataset.type = 'CocoDataset'
cfg.train_dataloader.dataset.data_root = cfg.data_root
cfg.train_dataloader.dataset.ann_file = train_anno_file
cfg.train_dataloader.dataset.data_prefix = dict(img='')
cfg.train_dataloader.dataset.metainfo = dict(from_file=metainfo_file)  # **关键修复**
cfg.train_dataloader.batch_size = BATCH_SIZE
cfg.train_dataloader.num_workers = min(4, os.cpu_count() // 2)
cfg.train_dataloader.pin_memory = True  # 加速数据加载

# 验证集
cfg.val_dataloader.dataset.type = 'CocoDataset'
cfg.val_dataloader.dataset.data_root = cfg.data_root
cfg.val_dataloader.dataset.ann_file = val_anno_file
cfg.val_dataloader.dataset.data_prefix = dict(img='')
cfg.val_dataloader.dataset.metainfo = dict(from_file=metainfo_file)  # **关键修复**
cfg.val_dataloader.batch_size = BATCH_SIZE
cfg.val_dataloader.num_workers = cfg.train_dataloader.num_workers

# 测试集
cfg.test_dataloader = cfg.val_dataloader

# 3. 评估器
cfg.val_evaluator = dict(
    type='CocoMetric',
    ann_file=val_anno_file,
    score_mode='keypoint'
)
cfg.test_evaluator = cfg.val_evaluator

# 4. 优化器与训练调度
cfg.optim_wrapper.optimizer.lr = LR
cfg.train_cfg.max_epochs = MAX_EPOCHS
cfg.train_cfg.val_interval = VAL_INTERVAL

# 修正学习率调度器的 epoch 范围
if 'param_scheduler' in cfg:
    for scheduler in cfg.param_scheduler:
        if scheduler.get('type') == 'CosineAnnealingLR':
            scheduler['T_max'] = MAX_EPOCHS
            scheduler['end'] = MAX_EPOCHS
        elif scheduler.get('type') == 'MultiStepLR':
            scheduler['milestones'] = [int(MAX_EPOCHS * 0.7), int(MAX_EPOCHS * 0.9)]

# 5. 日志与检查点
cfg.default_hooks.checkpoint.interval = VAL_INTERVAL
cfg.default_hooks.checkpoint.max_keep_ckpts = 3
cfg.default_hooks.logger.interval = 10

# 可视化
cfg.visualizer = dict(
    type='PoseLocalVisualizer',
    vis_backends=[dict(type='LocalVisBackend')],
    name='visualizer'
)

# 保存实际配置
os.makedirs(WORK_DIR, exist_ok=True)
cfg.dump(os.path.join(WORK_DIR, 'actual_config.py'))

print(f"\n✅ 配置完成:")
print(f"   Data Root: {cfg.data_root}")
print(f"   Train Annotations: {os.path.basename(train_anno_file)}")
print(f"   Val Annotations: {os.path.basename(val_anno_file)}")

✓ Metainfo 文件已定位: /home/tvem/anime/AniPose/mmpose/configs/_base_/datasets/coco.py

✅ 配置完成:
   Data Root: /home/tvem/anime/AniPose/68 model
   Train Annotations: anime_coco_train.json
   Val Annotations: anime_coco_val.json


# 微调

In [7]:
# 验证变量存在
assert 'train_anno_file' in globals(), "请先运行 Cell 2 生成数据"

# 加载基础配置
config_path = os.path.join('./mmpose/configs/body_2d_keypoint/rtmpose/coco', 
                          'rtmpose-t_8xb256-420e_coco-256x192.py')
cfg = Config.fromfile(config_path)

# 路径设置
mmpose_root = os.path.abspath('./mmpose/configs')
cfg.work_dir = WORK_DIR
cfg.data_root = os.path.abspath(BASE_DATA_DIR)

# ================= 关键修复：RTMCC 专用的数据管道 =================
# RTMCC使用SimCCLabel编码，不是MSRAHeatmap！

train_pipeline = [
    dict(type='LoadImage', file_client_args=dict(backend='disk')),
    dict(type='GetBBoxCenterScale'),
    dict(type='RandomFlip', direction='horizontal', prob=0.5),
    dict(type='RandomBBoxTransform', scale_factor=[0.75, 1.25], shift_factor=0.1),
    dict(type='TopdownAffine', input_size=INPUT_SIZE, use_udp=True),
    # 【关键修正】RTMCC使用SimCCLabel，不是MSRAHeatmap
    dict(type='GenerateTarget', encoder=dict(
        type='SimCCLabel',
        input_size=INPUT_SIZE,
        smoothing_type='gaussian',
        sigma=6.0,  # RTMPose-tiny典型配置
        simcc_split_ratio=2.0,  # 坐标分类的分割比例
        label_smooth_weight=0.0,
    )),
    dict(type='PackPoseInputs')
]

test_pipeline = [
    dict(type='LoadImage', file_client_args=dict(backend='disk')),
    dict(type='GetBBoxCenterScale'),
    dict(type='TopdownAffine', input_size=INPUT_SIZE, use_udp=True),
    dict(type='GenerateTarget', encoder=dict(
        type='SimCCLabel',
        input_size=INPUT_SIZE,
        smoothing_type='gaussian',
        sigma=6.0,
        simcc_split_ratio=2.0,
        label_smooth_weight=0.0,
    )),
    dict(type='PackPoseInputs')
]

# 数据加载器配置
cfg.train_dataloader = dict(
    batch_size=BATCH_SIZE,
    num_workers=4,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(
        type='CocoDataset',
        data_root=cfg.data_root,
        data_prefix=dict(img=''),
        ann_file=train_anno_file,
        metainfo=dict(from_file=os.path.join(mmpose_root, '_base_/datasets/coco.py')),
        pipeline=train_pipeline,
    )
)

cfg.val_dataloader = dict(
    batch_size=BATCH_SIZE,
    num_workers=4,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type='CocoDataset',
        data_root=cfg.data_root,
        data_prefix=dict(img=''),
        ann_file=val_anno_file,
        metainfo=dict(from_file=os.path.join(mmpose_root, '_base_/datasets/coco.py')),
        pipeline=test_pipeline,
        test_mode=True,
    )
)

cfg.test_dataloader = cfg.val_dataloader

# 评估器
cfg.val_evaluator = dict(type='CocoMetric', ann_file=val_anno_file, score_mode='keypoint', nms_mode='none')
cfg.test_evaluator = cfg.val_evaluator

# 优化器与训练设置（RTMPose使用AdamW）
cfg.optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=dict(type='AdamW', lr=LR, betas=(0.9, 0.999), weight_decay=0.01),
    paramwise_cfg=dict(
        norm_decay_mult=0,
        bias_decay_mult=0,
        bypass_duplicate=True,
    ),
)

cfg.train_cfg = dict(
    type='EpochBasedTrainLoop',
    max_epochs=MAX_EPOCHS,
    val_interval=VAL_INTERVAL,
)

# 学习率调度
cfg.param_scheduler = [
    dict(type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=500),
    dict(type='MultiStepLR', milestones=[int(MAX_EPOCHS*0.7), int(MAX_EPOCHS*0.9)], gamma=0.1),
]

# 日志与hooks
cfg.default_hooks = dict(
    timer=dict(type='IterTimerHook'),
    logger=dict(type='LoggerHook', interval=10),
    param_scheduler=dict(type='ParamSchedulerHook'),
    checkpoint=dict(type='CheckpointHook', interval=VAL_INTERVAL, save_best='coco/AP', rule='greater', max_keep_ckpts=3),
    sampler_seed=dict(type='DistSamplerSeedHook'),
)

cfg.visualizer = dict(type='PoseLocalVisualizer', vis_backends=[dict(type='LocalVisBackend')], name='visualizer')

# 确保模型head配置正确
cfg.model.head.decoder = dict(
    type='SimCCLabel',
    input_size=INPUT_SIZE,
    smoothing_type='gaussian',
    sigma=6.0,
    simcc_split_ratio=2.0,
)

# 启动训练
os.makedirs(WORK_DIR, exist_ok=True)
cfg.dump(os.path.join(WORK_DIR, 'config.py'))

print(f"配置完成: RTMPose-tiny + SimCCLabel编码")
print(f"  输入尺寸: {INPUT_SIZE}")
print(f"  训练样本: {len(json.load(open(train_anno_file))['images'])}")

runner = Runner.from_cfg(cfg)
print("\n🚀 开始训练...")
runner.train()

配置完成: RTMPose-tiny + SimCCLabel编码
  输入尺寸: (192, 256)
  训练样本: 306
03/20 11:06:29 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.4 | packaged by conda-forge | (default, Jul 17 2020, 15:16:46) [GCC 7.5.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 21
    GPU 0: NVIDIA GeForce RTX 2080 Ti
    CUDA_HOME: /usr
    NVCC: Cuda compilation tools, release 12.0, V12.0.140
    GCC: gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.4.2 (Git Hash 1137e04ec0b5251ca2b4400a4fd3c667ce843d67)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CU

/home/tvem/anaconda3/envs/mmpose/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmpose.visualization.local_visualizer.PoseLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


03/20 11:06:30 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
03/20 11:06:30 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
(NORMAL      ) PipelineSwitchHook                 
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng w

03/20 11:06:36 - mmengine - INFO - Epoch(train)  [1][10/39]  base_lr: 9.509018e-06 lr: 9.509018e-06  eta: 0:06:11  time: 0.320279  data_time: 0.145066  memory: 499  loss: 0.000502  loss_kpt: 0.000502  acc_pose: 0.000000


...

libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng w

03/20 11:11:42 - mmengine - INFO - Exp name: rtmpose-t_8xb256-420e_coco-256x192_20260320_110629
03/20 11:11:42 - mmengine - INFO - Saving checkpoint at 30 epochs


libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicatelibpng warning: eXIf: duplicate

libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
libpng warning: eXIf: duplicate
/home/tvem/anaconda3/envs/mmpose/lib/python3.8/site-packages/mmdet/models/layers/se_layer.py:158: FutureWarning: `torch.cuda.amp.autocas

03/20 11:11:44 - mmengine - INFO - Evaluating CocoMetric...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.02s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] =  0.688
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] =  1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] =  0.737
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] =  0.688
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] =  0.698
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] =  1.000
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] =  0.759
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = -

TopdownPoseEstimator(
  (data_preprocessor): PoseDataPreprocessor()
  (backbone): CSPNeXt(
    (stem): Sequential(
      (0): ConvModule(
        (conv): Conv2d(3, 12, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): _BatchNormXd(12, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): SiLU(inplace=True)
      )
      (1): ConvModule(
        (conv): Conv2d(12, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): _BatchNormXd(12, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): SiLU(inplace=True)
      )
      (2): ConvModule(
        (conv): Conv2d(12, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): _BatchNormXd(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): SiLU(inplace=True)
      )
    )
    (stage1): Sequential(
      (0): ConvModule(
        (conv): Conv2d(24, 48, kernel_size=(3, 3), str

# 结果可视化、对比Baseline